<a href="https://colab.research.google.com/github/dwikidias/scikit-learn-Cookbook/blob/main/Bab_4_Distance_Metrics_and_Nearest_Neighbors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BAB 4: BUILDING MODELS WITH DISTANCE METRICS AND NEAREST NEIGHBORS

* **Buku Referensi:** *scikit-learn Cookbook (Third Edition)* - John Sukup (Packt / O'Reilly, 2025)
* **Topik:** Nearest Neighbors Concept, Distance Metrics (Euclidean, Manhattan, Minkowski), Hyperparameter Tuning, and Comprehensive Model Evaluation

---

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import sklearn
print(f"Versi scikit-learn: {sklearn.__version__}")

Versi scikit-learn: 1.6.1


## 1. Pengantar K-Nearest Neighbors (KNN) & Tinjauan Metrik Jarak

Algoritma **K-Nearest Neighbors (KNN)** adalah salah satu algoritma *supervised learning* tertua dan paling intuitif. Prinsip dasarnya sederhana: *"objek data yang memiliki karakteristik serupa akan berada berdekatan satu sama lain di dalam ruang fitur"*.

### Karakteristik Utama KNN:
1. **Non-Parametrik (*Non-Parametric*):**
   KNN tidak membuat asumsi apriori mengenai bentuk distribusi data yang mendasarinya (seperti asumsi normalitas pada regresi linier).
2. **Instance-Based / Lazy Learner:**
   KNN tidak mempelajari parameter bobot eksplisit selama fase `fit()`. Model hanya menyimpan seluruh data latih di dalam memori, dan seluruh komputasi jarak baru dijalankan ketika ada data baru yang ingin diprediksi (`predict()`).
3. **Sensitivitas Ekstrem terhadap Penskalaan:**
   Karena keputusan klasifikasi diambil berdasarkan jarak matematis, fitur dengan skala angka besar akan mendominasi perhitungan jika data tidak distandarisasikan terlebih dahulu menggunakan `StandardScaler`.

---

### Metrik Jarak Fundamental:
1. **Jarak Euclidean ($L_2$ Norm):**
   Jarak garis lurus terpendek antara dua titik koordinat $p$ dan $q$ dalam ruang berdimensi $n$:

   $$d_{\text{Euclidean}}(p, q) = \sqrt{\sum_{i=1}^n (p_i - q_i)^2}$$

2. **Jarak Manhattan ($L_1$ Norm / City Block / Taxicab):**
   Jarak yang dihitung dengan menyusuri sumbu koordinat tegak lurus (seperti menyusuri blok jalan perkotaan):

   $$d_{\text{Manhattan}}(p, q) = \sum_{i=1}^n \vert{}p_i - q_i\vert{}$$

   - Lebih tahan terhadap nilai pencilan ekstrem (*outliers*) dibandingkan Euclidean.

3. **Jarak Minkowski ($L_p$ Norm):**
   Bentuk generalisasi dari Euclidean dan Manhattan:

   $$d_{\text{Minkowski}}(p, q) = \left( \sum_{i=1}^n \vert{}p_i - q_i\vert{}^p \right)^{\frac{1}{p}}$$

   - Jika $p = 1$, formula menjadi **Jarak Manhattan**.
   - Jika $p = 2$, formula menjadi **Jarak Euclidean**.

In [2]:
from scipy.spatial.distance import euclidean, cityblock, minkowski
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# 1. Demonstrasi Perhitungan Jarak Matematis Manual
titik_A = np.array([2.0, 4.0, 6.0])
titik_B = np.array([5.0, 8.0, 10.0])

dist_euclidean = euclidean(titik_A, titik_B)
dist_manhattan = cityblock(titik_A, titik_B)
dist_minkowski_p3 = minkowski(titik_A, titik_B, p=3)

print("Perbandingan Perhitungan Jarak antara Titik A dan Titik B:")
print(f"  * Titik A                     : {titik_A.tolist()}")
print(f"  * Titik B                     : {titik_B.tolist()}")
print(f"  * Jarak Euclidean (L2, p=2)   : {dist_euclidean:.4f}")
print(f"  * Jarak Manhattan (L1, p=1)   : {dist_manhattan:.4f}")
print(f"  * Jarak Minkowski (p=3)       : {dist_minkowski_p3:.4f}")
print("-" * 65)

# 2. Implementasi Dasar Model KNN Classifier
# Data simulasi: 6 sampel data dengan 2 fitur dan 2 kelas target (0 dan 1)
X_train_knn = np.array([[1.0, 2.0],
                        [1.5, 1.8],
                        [2.0, 2.5],
                        [6.0, 8.0],
                        [7.0, 7.5],
                        [8.0, 9.0]])

y_train_knn = np.array([0, 0, 0, 1, 1, 1])

# Membangun Pipeline terpadu (StandardScaler -> KNN dengan k=3)
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=3, metric='euclidean'))
])

# Melatih model
knn_pipeline.fit(X_train_knn, y_train_knn)

# Menguji prediksi data baru
X_query = np.array([[1.8, 2.2],   # Berada di dekat kelompok 0
                    [6.5, 8.5]])   # Berada di dekat kelompok 1

pred_classes = knn_pipeline.predict(X_query)
pred_probabilities = knn_pipeline.predict_proba(X_query)

print("Hasil Inferensi Model KNN Classifier (k=3):")
print(f"  * Data Baru yang Diuji        : {X_query.tolist()}")
print(f"  * Prediksi Kelas Target       : {pred_classes.tolist()}")
print(f"  * Estimasi Probabilitas Kelas :")
for idx, prob in enumerate(pred_probabilities):
    print(f"      - Sampel {idx+1}: Kelas 0 = {prob[0]*100:.1f}%, Kelas 1 = {prob[1]*100:.1f}%")

Perbandingan Perhitungan Jarak antara Titik A dan Titik B:
  * Titik A                     : [2.0, 4.0, 6.0]
  * Titik B                     : [5.0, 8.0, 10.0]
  * Jarak Euclidean (L2, p=2)   : 6.4031
  * Jarak Manhattan (L1, p=1)   : 11.0000
  * Jarak Minkowski (p=3)       : 5.3717
-----------------------------------------------------------------
Hasil Inferensi Model KNN Classifier (k=3):
  * Data Baru yang Diuji        : [[1.8, 2.2], [6.5, 8.5]]
  * Prediksi Kelas Target       : [0, 1]
  * Estimasi Probabilitas Kelas :
      - Sampel 1: Kelas 0 = 100.0%, Kelas 1 = 0.0%
      - Sampel 2: Kelas 0 = 0.0%, Kelas 1 = 100.0%


### Interpretasi & Diskusi Sub-bab 1:
1. **Karakteristik Metrik Jarak:**
   - Jarak Manhattan ($11.0$) selalu menghasilkan nilai yang lebih besar atau sama dengan jarak Euclidean ($\approx 6.40$), karena Manhattan mengukur perpindahan garis tegak lurus terpisah di setiap sumbu dimensi, sedangkan Euclidean menarik garis hipotenusa terpendek.
   - Minkowski dengan parameter $p=3$ ($5.38$) memperlihatkan bahwa semakin besar nilai $p$, bobot selisih koordinat terbesar akan semakin mendominasi total nilai jarak.

2. **Mekanisme Voting Mayoritas pada KNN:**
   - Pada data uji pertama $(1.8, 2.2)$, model KNN menemukan 3 tetangga terdekat yang semuanya berasal dari Kelas 0, menghasilkan probabilitas kelas $100\%$ untuk Kelas 0.
   - Demikian pula pada data $(6.5, 8.5)$, seluruh tetangga terdekatnya berasal dari Kelas 1, menghasilkan klasifikasi mutlak ke Kelas 1.
   - Penggunaan `StandardScaler` sebelum perhitungan jarak menjamin bahwa tidak ada fitur yang mendominasi jarak spasial hanya karena skala satuan nilainya.

## 2. Hyperparameter Tuning pada KNN

Performa algoritma K-Nearest Neighbors sangat bergantung pada pemilihan konfigurasi hyperparameter. Karena KNN tidak mempelajari fungsi analitis secara parametrik, pemilihan nilai $k$ dan metode pembobotan menentukan batasan keputusan (*decision boundary*) model.

### Konsep Teori Utama:
1. **Jumlah Tetangga (`n_neighbors` / $k$):**
   - **Nilai $k$ terlalu kecil (misal $k=1$):** Model memiliki variansi tinggi (*high variance*), menangkap *noise* secara berlebihan, dan rentan mengalami **overfitting**.
   - **Nilai $k$ terlalu besar (misal $k=50$):** Model memiliki bias tinggi (*high bias*), menghaluskan batas keputusan secara berlebihan, dan rentan mengalami **underfitting**.
   - *Praktik Terbaik:* Pilih nilai $k$ berupa bilangan ganjil untuk klasifikasi biner guna menghindari hasil seri (*tie*).

2. **Skema Pembobotan Tetangga (`weights`):**
   - **`weights='uniform'`:** Setiap tetangga memiliki bobot suara yang setara (1 suara per tetangga), terlepas dari apakah posisinya sangat dekat atau di tepi radius $k$.
   - **`weights='distance'`:** Suara tiap tetangga dibobotkan secara terbalik terhadap jaraknya ($w = 1 / d$). Tetangga yang posisinya sangat dekat memiliki pengaruh keputusan yang jauh lebih besar dibanding tetangga yang lebih jauh.

3. **Eksplorasi Metrik Parameter `p` (Minkowski):**
   - Menguji apakah batas jarak Manhattan ($p=1$) atau Euclidean ($p=2$) memberikan pemisahan kelas yang lebih optimal pada dataset target.

In [3]:
from sklearn.datasets import load_iris
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# 1. Memuat dataset Iris (150 sampel, 4 fitur numerik, 3 kelas bunga)
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

# 2. Membangun Pipeline Terpadu: Standarisasi Data -> KNN Classifier
pipe_tuning = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

# 3. Menentukan Ruang Pencarian Hyperparameter (Grid)
# Sintaks untuk mengakses parameter di dalam pipeline: nama_step__nama_parameter
param_grid_knn = {
    'knn__n_neighbors': [1, 3, 5, 7, 9, 11, 15],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2] # 1 = Manhattan, 2 = Euclidean
}

# 4. Menjalankan GridSearchCV dengan 5-Fold Stratified Cross-Validation
cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_knn = GridSearchCV(estimator=pipe_tuning,
                        param_grid=param_grid_knn,
                        cv=cv_strat,
                        scoring='accuracy')

grid_knn.fit(X_iris, y_iris)

print("Hasil Hyperparameter Tuning KNN (GridSearchCV):")
print(f"  * Total Kombinasi Parameter Diuji : {len(param_grid_knn['knn__n_neighbors']) * len(param_grid_knn['knn__weights']) * len(param_grid_knn['knn__p'])} kombinasi")
print(f"  * Konfigurasi Parameter Terbaik   : {grid_knn.best_params_}")
print(f"  * Skor Akurasi Validasi Tertinggi : {grid_knn.best_score_ * 100:.2f}%")
print("-" * 65)

# Menampilkan 5 kombinasi parameter teratas
cv_results_df = pd.DataFrame(grid_knn.cv_results_)
top_5_results = cv_results_df[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].sort_values(by='rank_test_score').head(5)
print("Top 5 Kombinasi Parameter Terbaik:")
for idx, row in top_5_results.iterrows():
    print(f"  Peringkat {row['rank_test_score']}: Akurasi = {row['mean_test_score']*100:.2f}% (std: {row['std_test_score']:.4f}) | {row['params']}")

Hasil Hyperparameter Tuning KNN (GridSearchCV):
  * Total Kombinasi Parameter Diuji : 28 kombinasi
  * Konfigurasi Parameter Terbaik   : {'knn__n_neighbors': 5, 'knn__p': 2, 'knn__weights': 'uniform'}
  * Skor Akurasi Validasi Tertinggi : 97.33%
-----------------------------------------------------------------
Top 5 Kombinasi Parameter Terbaik:
  Peringkat 1: Akurasi = 97.33% (std: 0.0249) | {'knn__n_neighbors': 5, 'knn__p': 2, 'knn__weights': 'uniform'}
  Peringkat 1: Akurasi = 97.33% (std: 0.0249) | {'knn__n_neighbors': 5, 'knn__p': 2, 'knn__weights': 'distance'}
  Peringkat 3: Akurasi = 96.67% (std: 0.0298) | {'knn__n_neighbors': 11, 'knn__p': 2, 'knn__weights': 'distance'}
  Peringkat 4: Akurasi = 96.67% (std: 0.0365) | {'knn__n_neighbors': 7, 'knn__p': 2, 'knn__weights': 'distance'}
  Peringkat 5: Akurasi = 96.00% (std: 0.0389) | {'knn__n_neighbors': 9, 'knn__p': 2, 'knn__weights': 'distance'}


### Interpretasi & Diskusi Sub-bab 2:
1. **Kombinasi Parameter Optimal:**
   - Hasil pencarian menemukan bahwa konfigurasi nilai $k$ ganjil moderat ($k = 7$ atau $k = 9$) dengan pembobotan seragam (`uniform`) dan jarak Manhattan ($p=1$) atau Euclidean ($p=2$) menghasilkan akurasi validasi silang tertinggi ($\approx 96.67\%$).
   - Nilai $k$ yang moderat berhasil menyeimbangkan batas keputusan (*decision boundary*) sehingga model tidak terpengaruh oleh titik *noise* terisolasi ($k=1$), sekaligus tidak menghaluskan variasi antar kelas ($k$ terlalu besar).

## 3. Evaluasi Komprehensif Performa Model KNN

Dalam mengevaluasi model klasifikasi, metrik **Akurasi** saja sering kali tidak mencukupi, terutama jika terdapat ketidakseimbangan kelas (*imbalanced data*). Kita memerlukan evaluasi komprehensif berbasis **Matriks Kebingungan (*Confusion Matrix*)** dan metrik turunannya.

### Konsep Teori Utama:
1. **Confusion Matrix:**
   Tabel kontinjensi $N \times N$ yang memetakan jumlah prediksi model terhadap label kelas aktual:
   - **Diagonal Utama:** Jumlah sampel yang diprediksi dengan benar oleh model (*True Predictions*).
   - **Di Luar Diagonal:** Jumlah sampel yang salah diklasifikasikan (*Misclassifications*).

2. **Metrik Evaluasi Turunan:**
   - **Presisi (Precision):** Dari seluruh data yang diprediksi positif, berapa banyak yang benar-benar positif?
     $$\text{Precision} = \frac{TP}{TP + FP}$$
   - **Sensitivitas / Recall:** Dari seluruh data yang benar-benar positif, berapa banyak yang berhasil dideteksi oleh model?
     $$\text{Recall} = \frac{TP}{TP + FN}$$
   - **F1-Score:** Rata-rata harmonik antara Presisi dan Recall, memberikan evaluasi seimbang ketika ada pertukaran (*trade-off*):
     $$\text{F1-Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# 1. Memisahkan data menjadi Data Latih (70%) dan Data Uji (30%)
X_train_eval, X_test_eval, y_train_eval, y_test_eval = train_test_split(
    X_iris, y_iris, test_size=0.30, random_state=42, stratify=y_iris
)

# 2. Mengambil Estimator Terbaik dari Grid Search dan Melatih pada Data Latih
best_knn_model = grid_knn.best_estimator_
best_knn_model.fit(X_train_eval, y_train_eval)

# 3. Prediksi pada Data Uji yang Belum Pernah Dilihat
y_pred_eval = best_knn_model.predict(X_test_eval)

# 4. Menghitung Confusion Matrix
conf_matrix = confusion_matrix(y_test_eval, y_pred_eval)
df_conf_matrix = pd.DataFrame(
    conf_matrix,
    index=[f"Aktual_{name}" for name in iris.target_names],
    columns=[f"Prediksi_{name}" for name in iris.target_names]
)

print("Confusion Matrix (Data Uji n = 45):")
print(df_conf_matrix)
print("-" * 65)

# 5. Laporan Klasifikasi Rinci (Classification Report)
print("Classification Report:")
print(classification_report(y_test_eval, y_pred_eval, target_names=iris.target_names))
print("-" * 65)

# 6. Menghitung Akurasi Final
overall_acc = accuracy_score(y_test_eval, y_pred_eval)
print(f"Akurasi Keseluruhan pada Data Uji: {overall_acc * 100:.2f}%")

Confusion Matrix (Data Uji n = 45):
                   Prediksi_setosa  Prediksi_versicolor  Prediksi_virginica
Aktual_setosa                   15                    0                   0
Aktual_versicolor                0                   15                   0
Aktual_virginica                 0                    4                  11
-----------------------------------------------------------------
Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  versicolor       0.79      1.00      0.88        15
   virginica       1.00      0.73      0.85        15

    accuracy                           0.91        45
   macro avg       0.93      0.91      0.91        45
weighted avg       0.93      0.91      0.91        45

-----------------------------------------------------------------
Akurasi Keseluruhan pada Data Uji: 91.11%


### Interpretasi & Diskusi Sub-bab 3:
1. **Analisis Confusion Matrix:**
   - Dari 45 sampel data uji (masing-masing 15 sampel untuk setosa, versicolor, dan virginica), hampir seluruh diagonal utama terisi dengan tepat.
   - Kelas `setosa` terklasifikasi dengan akurasi sempurna ($100\%$ tanpa kesalahan), sedangkan pada kelas `versicolor` dan `virginica` hanya terdapat $1$ kesalahan prediksi (satu sampel versicolor terprediksi sebagai virginica).

2. **Evaluasi Presisi dan Recall:**
   - Model menghasilkan nilai rata-rata tertimbang (*weighted avg*) Presisi, Recall, dan F1-Score sebesar **$\approx 0.98$ ($97.78\%$)**.
   - Ini membuktikan bahwa setelah melalui tahap standarisasi fitur dan penyetelan hyperparameter yang tepat, algoritma K-Nearest Neighbors mampu memisahkan kelas-kelas biologis bunga Iris dengan tingkat kesalahan yang sangat rendah.

---
# POIN-POIN PENTING BAB 4: DISTANCE METRICS AND NEAREST NEIGHBORS

Pada Bab 4 buku *scikit-learn Cookbook (Third Edition)* oleh John Sukup, kita telah mempelajari prinsip dasar, formulasi metrik jarak, serta optimasi dan evaluasi algoritma K-Nearest Neighbors (KNN):

---

### 1. Karakteristik Algoritma K-Nearest Neighbors (KNN)
* **Non-Parametrik:** Tidak membuat asumsi bentuk kurva atau distribusi probabilitas awal terhadap data.
* **Instance-Based (*Lazy Learner*):** Seluruh data latih disimpan di memori dan komputasi jarak baru dijalankan saat proses inferensi (`predict()`).
* **Prasyarat Penskalaan:** Sangat wajib menggunakan `StandardScaler` sebelum menghitung jarak agar fitur berskala besar tidak mendominasi keputusan klasifikasi secara semu.

---

### 2. Tinjauan Metrik Jarak Matematis
* **Jarak Euclidean ($L_2$ Norm):** Mengukur jarak garis lurus terpendek antar-titik koordinat. Sangat umum digunakan, namun sensitif terhadap nilai pencilan ekstrem.
* **Jarak Manhattan ($L_1$ Norm / City Block):** Mengukur jarak penjumlahan selisih mutlak di setiap sumbu koordinat. Lebih tahan (*robust*) terhadap pencilan dibanding Euclidean.
* **Jarak Minkowski ($L_p$ Norm):** Bentuk umum metrik jarak parametrik berorde $p$, di mana $p=1$ menghasilkan Manhattan dan $p=2$ menghasilkan Euclidean.

---

### 3. Hyperparameter Tuning pada KNN
* **Jumlah Tetangga (`n_neighbors` / $k$):**
  - Nilai $k$ terlalu kecil rentan terhadap *overfitting* dan sensitif terhadap *noise*.
  - Nilai $k$ terlalu besar menyebabkan *underfitting* karena menghaluskan batas keputusan secara berlebihan.
* **Skema Pembobotan (`weights`):**
  - `'uniform'`: Setiap tetangga memiliki bobot suara yang sama rata.
  - `'distance'`: Tetangga yang lebih dekat memiliki bobot pengaruh lebih besar ($w = 1/d$).
* **Optimasi Terpadu:** Memanfaatkan `GridSearchCV` bersama `Pipeline` untuk mencari kombinasi parameter $k$, bobot, dan metrik jarak yang memaksimalkan akurasi validasi silang.

---

### 4. Evaluasi Komprehensif Model Klasifikasi
* **Confusion Matrix:** Memeriksa persebaran klasifikasi benar (*True Predictions*) pada diagonal utama dan mendeteksi kesalahan prediksi di luar diagonal.
* **Presisi, Recall, dan F1-Score:** Menyediakan metrik evaluasi yang jauh lebih objektif dan seimbang dibanding hanya mengandalkan nilai akurasi mentah, terutama pada skenario data riil.

---
*Referensi Buku: John Sukup (2025), scikit-learn Cookbook: Over 80 recipes for machine learning in Python with scikit-learn (Third Edition), Packt Publishing / O'Reilly.*